In [0]:
# Cell 1: UI contract views setup + source table preflight
# Notebook: 34_ui_contract_views
#
# Purpose:
# - Promote tested UI adapter views into a permanent notebook
# - Validate source Gold tables exist before creating ui_* views
# - Keep raw Gold marts as source of truth
#
# Serverless-safe:
# - No cache()
# - No persist()
# - No CACHE TABLE
# - No restricted Spark configs

from pyspark.sql import functions as F

GOLD_SCHEMA = "supplysage_gold"
UI_CONTRACT_NOTEBOOK = "34_ui_contract_views"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")

source_table_checks = [
    {
        "source_table": f"{GOLD_SCHEMA}.gold_dashboard_supplier_risk_summary",
        "purpose": "Command Center supplier risk cards",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "overall_risk_score",
            "risk_band",
            "active_event_count_30d",
            "impacted_sku_count",
            "top_risk_driver",
            "recommended_action",
        ],
    },
    {
        "source_table": f"{GOLD_SCHEMA}.gold_dashboard_sku_stockout_summary",
        "purpose": "Command Center SKU stockout cards",
        "required_columns": [
            "canonical_sku_id",
            "stockout_probability",
            "stockout_risk_band",
            "days_of_cover",
            "sales_exposure_30d",
        ],
    },
    {
        "source_table": f"{GOLD_SCHEMA}.gold_dim_suppliers",
        "purpose": "Supplier 360 profile/header",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "country",
            "region",
            "supplier_category",
            "criticality_tier",
            "annual_spend",
            "single_source_flag",
        ],
    },
    {
        "source_table": f"{GOLD_SCHEMA}.gold_supplier_risk_scores",
        "purpose": "Supplier 360 risk score and drivers",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "overall_risk_score",
            "risk_band",
            "top_risk_driver",
            "recommended_action",
            "operational_score",
            "dependency_score",
            "external_event_score",
            "logistics_score",
            "sanctions_score",
            "cyber_score",
        ],
    },
    {
        "source_table": f"{GOLD_SCHEMA}.gold_supplier_risk_agent_run_logs",
        "purpose": "Agent/LLM run logs",
        "required_columns": [
            "agent_run_id",
            "supplier_id",
            "question",
            "final_answer",
            "ok",
            "recommended_action",
            "evidence_count",
        ],
    },
    {
        "source_table": f"{GOLD_SCHEMA}.gold_alert_inbox",
        "purpose": "Operational alert inbox",
        "required_columns": [
            "alert_id",
            "severity",
            "supplier_id",
            "supplier_name",
            "sku_id",
            "inbox_status",
        ],
    },
]

def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False

preflight_rows = []

for item in source_table_checks:
    source_table = item["source_table"]
    exists = table_exists(source_table)

    if exists:
        cols = spark.table(source_table).columns
        missing_columns = [c for c in item["required_columns"] if c not in cols]
        status = "READY" if not missing_columns else "MISSING_COLUMNS"
    else:
        cols = []
        missing_columns = item["required_columns"]
        status = "MISSING_TABLE"

    preflight_rows.append({
        "source_table": source_table,
        "purpose": item["purpose"],
        "exists": exists,
        "status": status,
        "missing_columns": ", ".join(missing_columns),
        "column_count": len(cols),
    })

preflight_df = spark.createDataFrame(preflight_rows)

display(
    preflight_df.orderBy(
        F.when(F.col("status") == "MISSING_TABLE", 0)
         .when(F.col("status") == "MISSING_COLUMNS", 1)
         .otherwise(2),
        "source_table",
    )
)

failed_count = preflight_df.filter(F.col("status") != "READY").count()

print("UI contract source preflight summary")
print(f"READY: {preflight_df.filter(F.col('status') == 'READY').count()}")
print(f"FAILED: {failed_count}")

if failed_count > 0:
    raise ValueError("Source preflight failed. Fix missing source tables/columns before creating UI contract views.")

print("✅ Source preflight passed. Ready to create permanent ui_* adapter views.")

In [0]:
# Cell 2: Create permanent SupplySage UI contract views
# Notebook: 34_ui_contract_views
#
# Purpose:
# - Create frontend-facing ui_* views from existing Gold source tables
# - Add stable UI aliases without rewriting Gold marts
# - These views are the contract layer for the Next.js data adapter/API layer
#
# Serverless-safe:
# - CREATE OR REPLACE VIEW only
# - No cache()
# - No persist()
# - No heavy full-table operations

GOLD_SCHEMA = "supplysage_gold"
UI_CONTRACT_VERSION = "ui_contract_v1"

# -------------------------------------------------------------------
# 1. Command Center supplier risk summary
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_command_center_supplier_risk_summary AS
SELECT
    *,
    overall_risk_score AS supplier_risk_score,
    top_risk_driver AS final_top_risk_driver,
    recommended_action AS final_recommended_action,
    '{UI_CONTRACT_VERSION}' AS ui_contract_version
FROM {GOLD_SCHEMA}.gold_dashboard_supplier_risk_summary
""")


# -------------------------------------------------------------------
# 2. Command Center SKU stockout summary
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_command_center_sku_stockout_summary AS
SELECT
    *,
    CASE
        WHEN stockout_probability IS NULL THEN NULL
        WHEN stockout_probability <= 1 THEN ROUND(stockout_probability * 100, 2)
        ELSE ROUND(stockout_probability, 2)
    END AS sku_stockout_risk_score,
    sales_exposure_30d AS revenue_at_risk,
    '{UI_CONTRACT_VERSION}' AS ui_contract_version
FROM {GOLD_SCHEMA}.gold_dashboard_sku_stockout_summary
""")


# -------------------------------------------------------------------
# 3. Supplier 360 profile/header
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_dim_suppliers AS
SELECT
    *,
    supplier_category AS category,
    '{UI_CONTRACT_VERSION}' AS ui_contract_version
FROM {GOLD_SCHEMA}.gold_dim_suppliers
""")


# -------------------------------------------------------------------
# 4. Supplier 360 risk score and drivers
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_supplier_risk_scores AS
SELECT
    *,
    overall_risk_score AS supplier_risk_score,
    '{UI_CONTRACT_VERSION}' AS ui_contract_version
FROM {GOLD_SCHEMA}.gold_supplier_risk_scores
""")


# -------------------------------------------------------------------
# 5. Agent / LLM run logs
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_supplier_risk_agent_run_logs AS
SELECT
    *,
    agent_run_id AS run_id,
    final_answer AS answer,
    '{UI_CONTRACT_VERSION}' AS ui_contract_version
FROM {GOLD_SCHEMA}.gold_supplier_risk_agent_run_logs
""")


# -------------------------------------------------------------------
# 6. Alert inbox operational UI contract
# -------------------------------------------------------------------

spark.sql(f"""
CREATE OR REPLACE VIEW {GOLD_SCHEMA}.ui_alert_inbox AS
SELECT
    *,
    CASE
        WHEN inbox_status IS NULL THEN 'New'
        WHEN lower(inbox_status) IN ('new', 'open', 'created', 'drafted') THEN 'New'
        WHEN lower(inbox_status) IN ('acknowledged', 'ack') THEN 'Acknowledged'
        WHEN lower(inbox_status) IN ('investigating', 'in_progress', 'in progress') THEN 'Investigating'
        WHEN lower(inbox_status) IN ('mitigated') THEN 'Mitigated'
        WHEN lower(inbox_status) IN ('resolved', 'closed') THEN 'Resolved'
        ELSE initcap(inbox_status)
    END AS status,

    CASE
        WHEN supplier_id IS NOT NULL THEN 'supplier'
        WHEN sku_id IS NOT NULL THEN 'sku'
        ELSE 'supplier'
    END AS entity_type,

    CAST(coalesce(supplier_id, sku_id, alert_id) AS STRING) AS entity_id,

    CAST(coalesce(supplier_name, sku_id, alert_id) AS STRING) AS entity_name,

    CASE
        WHEN channel IS NULL THEN 'Dashboard Inbox'
        WHEN lower(channel) = 'email' THEN 'Email'
        WHEN lower(channel) IN ('dashboard', 'dashboard inbox') THEN 'Dashboard Inbox'
        WHEN lower(channel) IN ('email + dashboard', 'email + dashboard inbox') THEN 'Email + Dashboard Inbox'
        ELSE channel
    END AS delivery_channel,

    '{UI_CONTRACT_VERSION}' AS ui_contract_version
FROM {GOLD_SCHEMA}.gold_alert_inbox
""")


# -------------------------------------------------------------------
# 7. Validate permanent views
# -------------------------------------------------------------------

ui_view_checks = [
    {
        "view_name": f"{GOLD_SCHEMA}.ui_command_center_supplier_risk_summary",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "supplier_risk_score",
            "risk_band",
            "active_event_count_30d",
            "impacted_sku_count",
            "final_top_risk_driver",
            "final_recommended_action",
            "ui_contract_version",
        ],
    },
    {
        "view_name": f"{GOLD_SCHEMA}.ui_command_center_sku_stockout_summary",
        "required_columns": [
            "canonical_sku_id",
            "sku_stockout_risk_score",
            "stockout_risk_band",
            "days_of_cover",
            "revenue_at_risk",
            "ui_contract_version",
        ],
    },
    {
        "view_name": f"{GOLD_SCHEMA}.ui_dim_suppliers",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "country",
            "region",
            "category",
            "criticality_tier",
            "annual_spend",
            "single_source_flag",
            "ui_contract_version",
        ],
    },
    {
        "view_name": f"{GOLD_SCHEMA}.ui_supplier_risk_scores",
        "required_columns": [
            "supplier_id",
            "supplier_name",
            "supplier_risk_score",
            "risk_band",
            "top_risk_driver",
            "recommended_action",
            "operational_score",
            "dependency_score",
            "external_event_score",
            "logistics_score",
            "sanctions_score",
            "cyber_score",
            "ui_contract_version",
        ],
    },
    {
        "view_name": f"{GOLD_SCHEMA}.ui_supplier_risk_agent_run_logs",
        "required_columns": [
            "run_id",
            "supplier_id",
            "question",
            "answer",
            "ok",
            "recommended_action",
            "evidence_count",
            "ui_contract_version",
        ],
    },
    {
        "view_name": f"{GOLD_SCHEMA}.ui_alert_inbox",
        "required_columns": [
            "alert_id",
            "severity",
            "status",
            "entity_type",
            "entity_id",
            "entity_name",
            "delivery_channel",
            "ui_contract_version",
        ],
    },
]

validation_rows = []

for item in ui_view_checks:
    view_name = item["view_name"]
    df = spark.table(view_name)
    cols = df.columns
    missing_columns = [c for c in item["required_columns"] if c not in cols]

    validation_rows.append({
        "view_name": view_name,
        "status": "READY" if not missing_columns else "FAILED",
        "missing_columns": ", ".join(missing_columns),
        "column_count": len(cols),
    })

validation_df = spark.createDataFrame(validation_rows)

display(validation_df.orderBy("view_name"))

ready_count = validation_df.filter("status = 'READY'").count()
failed_count = validation_df.filter("status != 'READY'").count()

print("Permanent UI contract view creation summary")
print(f"READY: {ready_count}")
print(f"FAILED: {failed_count}")

if failed_count > 0:
    raise ValueError("One or more UI contract views failed validation.")

print("✅ Permanent ui_* contract views are created and validated.")
print("Next step: create a UI contract registry table that maps frontend routes to these views.")

In [0]:
# Cell 3: Create permanent UI contract registry table
# Notebook: 34_ui_contract_views
#
# Purpose:
# - Create a single registry table that maps frontend routes to Databricks UI contracts
# - Make it easy for frontend/backend API developers to know which view/table powers each page
# - Keep this as documentation + machine-readable contract metadata
#
# Serverless-safe:
# - Small static table write only
# - No cache()
# - No persist()

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
)

GOLD_SCHEMA = "supplysage_gold"
UI_CONTRACT_VERSION = "ui_contract_v1"
REGISTRY_TABLE = f"{GOLD_SCHEMA}.ui_frontend_contract_registry"

registry_schema = StructType([
    StructField("route", StringType(), False),
    StructField("page_name", StringType(), False),
    StructField("contract_name", StringType(), False),
    StructField("contract_type", StringType(), False),
    StructField("contract_role", StringType(), False),
    StructField("adapter_function", StringType(), False),
    StructField("uses_agent_context", BooleanType(), False),
    StructField("is_primary_contract", BooleanType(), False),
    StructField("frontend_priority", IntegerType(), False),
    StructField("notes", StringType(), True),
])

registry_rows = [
    {
        "route": "/",
        "page_name": "Landing Page",
        "contract_name": "static_frontend_content",
        "contract_type": "frontend_static",
        "contract_role": "Project website hero, business case, impact cards, architecture preview, and demo CTA",
        "adapter_function": "getLandingPageData",
        "uses_agent_context": False,
        "is_primary_contract": True,
        "frontend_priority": 1,
        "notes": "Use static content plus demo-safe project metrics. Do not redirect / to /command-center.",
    },
    {
        "route": "/command-center",
        "page_name": "Command Center",
        "contract_name": f"{GOLD_SCHEMA}.ui_command_center_supplier_risk_summary",
        "contract_type": "view",
        "contract_role": "Supplier risk cards, ranked supplier table, executive KPIs",
        "adapter_function": "getCommandCenterSupplierRisk",
        "uses_agent_context": False,
        "is_primary_contract": True,
        "frontend_priority": 1,
        "notes": "Use this instead of raw gold_dashboard_supplier_risk_summary.",
    },
    {
        "route": "/command-center",
        "page_name": "Command Center",
        "contract_name": f"{GOLD_SCHEMA}.ui_command_center_sku_stockout_summary",
        "contract_type": "view",
        "contract_role": "SKU stockout exposure cards and impacted SKU table",
        "adapter_function": "getCommandCenterSkuStockout",
        "uses_agent_context": False,
        "is_primary_contract": False,
        "frontend_priority": 2,
        "notes": "Use virtualization for large SKU tables.",
    },
    {
        "route": "/command-center",
        "page_name": "Command Center",
        "contract_name": f"{GOLD_SCHEMA}.gold_external_risk_event_mart",
        "contract_type": "table",
        "contract_role": "External risk event feed",
        "adapter_function": "getExternalRiskEvents",
        "uses_agent_context": False,
        "is_primary_contract": False,
        "frontend_priority": 3,
        "notes": "Limit results in API layer. Do not render thousands of events at once.",
    },
    {
        "route": "/command-center",
        "page_name": "Command Center",
        "contract_name": f"{GOLD_SCHEMA}.gold_alerting_ui_breakdown",
        "contract_type": "table",
        "contract_role": "Alert summary cards and breakdown metrics",
        "adapter_function": "getAlertingBreakdown",
        "uses_agent_context": False,
        "is_primary_contract": False,
        "frontend_priority": 4,
        "notes": "Use for severity/status/channel summary cards.",
    },
    {
        "route": "/alerts",
        "page_name": "Alert Inbox",
        "contract_name": f"{GOLD_SCHEMA}.ui_alert_inbox",
        "contract_type": "view",
        "contract_role": "Operational alert inbox with workflow status and entity fields",
        "adapter_function": "getAlerts",
        "uses_agent_context": True,
        "is_primary_contract": True,
        "frontend_priority": 1,
        "notes": "Frontend must not send real email. Show preview/mock delivery only.",
    },
    {
        "route": "/alerts/[alertId]",
        "page_name": "Alert Detail",
        "contract_name": f"{GOLD_SCHEMA}.ui_alert_inbox",
        "contract_type": "view",
        "contract_role": "Alert summary, trigger, owner, recipients, status, and recommended action",
        "adapter_function": "getAlertDetail",
        "uses_agent_context": True,
        "is_primary_contract": True,
        "frontend_priority": 1,
        "notes": "Filter by alert_id in API layer.",
    },
    {
        "route": "/alerts/[alertId]",
        "page_name": "Alert Detail",
        "contract_name": f"{GOLD_SCHEMA}.gold_alert_delivery_log",
        "contract_type": "table",
        "contract_role": "Delivery status and email notification history",
        "adapter_function": "getAlertDeliveryLog",
        "uses_agent_context": False,
        "is_primary_contract": False,
        "frontend_priority": 2,
        "notes": "Use only mock/demo-safe delivery status in frontend.",
    },
    {
        "route": "/alerts/[alertId]",
        "page_name": "Alert Detail",
        "contract_name": f"{GOLD_SCHEMA}.gold_rag_evidence_chunks",
        "contract_type": "table",
        "contract_role": "Evidence cards for why alert triggered",
        "adapter_function": "getAlertEvidence",
        "uses_agent_context": True,
        "is_primary_contract": False,
        "frontend_priority": 3,
        "notes": "Join/filter by supplier_id or alert-linked entity context in API layer.",
    },
    {
        "route": "/suppliers/[supplierId]",
        "page_name": "Supplier 360",
        "contract_name": f"{GOLD_SCHEMA}.ui_dim_suppliers",
        "contract_type": "view",
        "contract_role": "Supplier profile header",
        "adapter_function": "getSupplierProfile",
        "uses_agent_context": False,
        "is_primary_contract": True,
        "frontend_priority": 1,
        "notes": "Filter by supplier_id.",
    },
    {
        "route": "/suppliers/[supplierId]",
        "page_name": "Supplier 360",
        "contract_name": f"{GOLD_SCHEMA}.ui_supplier_risk_scores",
        "contract_type": "view",
        "contract_role": "Risk score, risk band, score breakdown, drivers, and recommended action",
        "adapter_function": "getSupplierRiskScores",
        "uses_agent_context": False,
        "is_primary_contract": False,
        "frontend_priority": 2,
        "notes": "Use this instead of raw gold_supplier_risk_scores.",
    },
    {
        "route": "/suppliers/[supplierId]",
        "page_name": "Supplier 360",
        "contract_name": f"{GOLD_SCHEMA}.gold_supplier_sku_dependency_mart",
        "contract_type": "table",
        "contract_role": "SKU Dependency tab and React Flow graph",
        "adapter_function": "getSupplierSkuDependency",
        "uses_agent_context": False,
        "is_primary_contract": False,
        "frontend_priority": 3,
        "notes": "Transform supplier/SKU rows into React Flow nodes and edges.",
    },
    {
        "route": "/suppliers/[supplierId]",
        "page_name": "Supplier 360",
        "contract_name": f"{GOLD_SCHEMA}.gold_supplier_performance_mart",
        "contract_type": "table",
        "contract_role": "Performance tab metrics and delivery trends",
        "adapter_function": "getSupplierPerformance",
        "uses_agent_context": False,
        "is_primary_contract": False,
        "frontend_priority": 4,
        "notes": "Use fill rate, on-time delivery, quality, defects, lead time, and delay trend.",
    },
    {
        "route": "/suppliers/[supplierId]",
        "page_name": "Supplier 360",
        "contract_name": f"{GOLD_SCHEMA}.gold_external_risk_event_mart",
        "contract_type": "table",
        "contract_role": "Latest external events and evidence context",
        "adapter_function": "getSupplierExternalEvents",
        "uses_agent_context": True,
        "is_primary_contract": False,
        "frontend_priority": 5,
        "notes": "Filter by supplier_id when available; otherwise by supplier/entity matching context.",
    },
    {
        "route": "/suppliers/[supplierId]",
        "page_name": "Supplier 360",
        "contract_name": f"{GOLD_SCHEMA}.gold_chat_context_snapshots",
        "contract_type": "table",
        "contract_role": "Preassembled supplier context for chat panel and agent prompt grounding",
        "adapter_function": "getSupplierChatContext",
        "uses_agent_context": True,
        "is_primary_contract": False,
        "frontend_priority": 6,
        "notes": "Chat reads this first before RAG.",
    },
    {
        "route": "/suppliers/[supplierId]",
        "page_name": "Supplier 360",
        "contract_name": f"{GOLD_SCHEMA}.ui_supplier_risk_agent_run_logs",
        "contract_type": "view",
        "contract_role": "Agent Investigation tab run history and final answers",
        "adapter_function": "getSupplierAgentRuns",
        "uses_agent_context": True,
        "is_primary_contract": False,
        "frontend_priority": 7,
        "notes": "Use run_id and answer aliases.",
    },
    {
        "route": "/suppliers/[supplierId]/skus",
        "page_name": "Supplier Impacted SKUs",
        "contract_name": f"{GOLD_SCHEMA}.ui_command_center_sku_stockout_summary",
        "contract_type": "view",
        "contract_role": "Impacted SKUs sorted by urgency",
        "adapter_function": "getSupplierImpactedSkus",
        "uses_agent_context": False,
        "is_primary_contract": True,
        "frontend_priority": 1,
        "notes": "Filter by supplier_id, sort by stockout_risk_score and revenue_at_risk.",
    },
    {
        "route": "/lineage",
        "page_name": "Lineage & Trust",
        "contract_name": "static_frontend_content",
        "contract_type": "frontend_static",
        "contract_role": "Medallion pipeline, source cards, volume metrics, validation summary, trust callouts",
        "adapter_function": "getLineagePageData",
        "uses_agent_context": False,
        "is_primary_contract": True,
        "frontend_priority": 1,
        "notes": "Use project constants from the outline unless separate lineage tables are added later.",
    },
    {
        "route": "/demo",
        "page_name": "Guided Demo",
        "contract_name": "static_frontend_content",
        "contract_type": "frontend_static",
        "contract_role": "Guided story from disruption to supplier risk to SKU exposure to alert",
        "adapter_function": "getGuidedDemoData",
        "uses_agent_context": True,
        "is_primary_contract": True,
        "frontend_priority": 1,
        "notes": "Use curated demo-safe scenario. This should feel like a product walkthrough.",
    },
    {
        "route": "/api/agent/query",
        "page_name": "Agent Query API",
        "contract_name": f"{GOLD_SCHEMA}.gold_chat_context_snapshots",
        "contract_type": "table",
        "contract_role": "Fast supplier context lookup before RAG/LLM",
        "adapter_function": "runSupplierRiskAgent",
        "uses_agent_context": True,
        "is_primary_contract": True,
        "frontend_priority": 1,
        "notes": "Never expose Databricks or LLM credentials in browser code.",
    },
    {
        "route": "/api/agent/query",
        "page_name": "Agent Query API",
        "contract_name": f"{GOLD_SCHEMA}.gold_rag_retrieval_index",
        "contract_type": "table",
        "contract_role": "RAG retrieval metadata index",
        "adapter_function": "retrieveSupplierEvidence",
        "uses_agent_context": True,
        "is_primary_contract": False,
        "frontend_priority": 2,
        "notes": "Use server-side only.",
    },
    {
        "route": "/api/agent/query",
        "page_name": "Agent Query API",
        "contract_name": f"{GOLD_SCHEMA}.ui_supplier_risk_agent_run_logs",
        "contract_type": "view",
        "contract_role": "Agent response/run log display",
        "adapter_function": "getAgentRunLogs",
        "uses_agent_context": True,
        "is_primary_contract": False,
        "frontend_priority": 3,
        "notes": "Use for UI history and observability.",
    },
]

registry_df = (
    spark.createDataFrame(registry_rows, schema=registry_schema)
    .withColumn("ui_contract_version", F.lit(UI_CONTRACT_VERSION))
    .withColumn("registry_created_at", F.current_timestamp())
    .withColumn("registry_created_by", F.lit("34_ui_contract_views"))
)

(
    registry_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REGISTRY_TABLE)
)

check_df = spark.table(REGISTRY_TABLE)

display(
    check_df
    .orderBy("route", "frontend_priority", "contract_name")
)

print("UI frontend contract registry summary")
print(f"Registry table: {REGISTRY_TABLE}")
print(f"Rows: {check_df.count()}")
print(f"Routes: {check_df.select('route').distinct().count()}")
print("✅ UI frontend contract registry created.")
print("Next step: create final contract smoke test cell.")

In [0]:
# Cell 4: Final UI contract smoke test
# Notebook: 34_ui_contract_views
#
# Purpose:
# - Smoke test all real Databricks contracts listed in ui_frontend_contract_registry
# - Skip frontend_static contracts because those are implemented in Next.js
# - Confirm every real view/table is queryable and has at least a basic schema
#
# Serverless-safe:
# - Uses limit(1) for queryability checks
# - No cache()
# - No persist()
# - No full counts on large tables

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, IntegerType

GOLD_SCHEMA = "supplysage_gold"
REGISTRY_TABLE = f"{GOLD_SCHEMA}.ui_frontend_contract_registry"

registry_df = spark.table(REGISTRY_TABLE)

real_contracts_df = (
    registry_df
    .filter(~F.col("contract_type").isin("frontend_static"))
    .select(
        "route",
        "page_name",
        "contract_name",
        "contract_type",
        "contract_role",
        "adapter_function",
        "is_primary_contract",
        "frontend_priority",
    )
    .dropDuplicates(["contract_name"])
    .orderBy("contract_name")
)

real_contracts = real_contracts_df.collect()

smoke_rows = []

print("Running UI contract smoke test...")
print("=" * 90)

for row in real_contracts:
    contract_name = row["contract_name"]

    try:
        df = spark.table(contract_name)
        sample_rows = df.limit(1).collect()
        cols = df.columns

        status = "PASS"
        is_queryable = True
        error_message = None
        column_count = len(cols)
        has_sample_row = len(sample_rows) > 0

    except Exception as e:
        status = "FAIL"
        is_queryable = False
        error_message = str(e)[:1000]
        column_count = 0
        has_sample_row = False

    smoke_rows.append({
        "contract_name": contract_name,
        "contract_type": row["contract_type"],
        "example_route": row["route"],
        "example_page_name": row["page_name"],
        "adapter_function": row["adapter_function"],
        "is_primary_contract": row["is_primary_contract"],
        "frontend_priority": row["frontend_priority"],
        "status": status,
        "is_queryable": is_queryable,
        "column_count": column_count,
        "has_sample_row": has_sample_row,
        "error_message": error_message,
    })

smoke_schema = StructType([
    StructField("contract_name", StringType(), False),
    StructField("contract_type", StringType(), True),
    StructField("example_route", StringType(), True),
    StructField("example_page_name", StringType(), True),
    StructField("adapter_function", StringType(), True),
    StructField("is_primary_contract", BooleanType(), True),
    StructField("frontend_priority", IntegerType(), True),
    StructField("status", StringType(), False),
    StructField("is_queryable", BooleanType(), False),
    StructField("column_count", IntegerType(), False),
    StructField("has_sample_row", BooleanType(), False),
    StructField("error_message", StringType(), True),
])

smoke_df = spark.createDataFrame(smoke_rows, schema=smoke_schema)

display(
    smoke_df.orderBy(
        F.when(F.col("status") == "FAIL", 0).otherwise(1),
        "contract_name",
    )
)

pass_count = smoke_df.filter("status = 'PASS'").count()
fail_count = smoke_df.filter("status = 'FAIL'").count()

print("Final UI contract smoke test summary")
print(f"PASS: {pass_count}")
print(f"FAIL: {fail_count}")

if fail_count > 0:
    raise ValueError("One or more frontend backend contracts failed the smoke test.")

print("✅ All real Databricks UI contracts are queryable.")
print("✅ 34_ui_contract_views is ready.")
print("Next step: start the Next.js frontend and use ui_frontend_contract_registry as the source-of-truth route map.")